### Transform Results Data
1. Read bronze results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorld → constructor_id, driverld → driver_id, racelane → race_name, positionText → finish_position_text )
4. Rename columns to make them more meaningful (date → race_date, grid → grid position, laps → completed_laps, number → car_number, position → finish_position)
5. Filter out rows where season, round, custructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_nane to Title Case
8. Write the transformed data to silver results table

In [0]:
from pyspark.sql import functions as f

Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
result_df=(
     spark.read.table(bronze_table)
     .select(
         "constructorId",
         "date",
         "driverId",
         "grid",
         "laps",
         "number",
         "points",
         "position",
         "positionText",
         "raceName",
         "round",
         "season",
         "status",
         "ingestion_timestamp",
         "source_file"
     )
     .withColumnsRenamed(
        {
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName":"race_name",
            "positionText":"finish_position_text",
            "date":"race_date",
            "grid":"grid_position",
            "laps":"compeleted_laps",
            "number":"car_number",
            "position":"finish_position"
        }
    )
)

In [0]:
result_valid_df=(
    result_df
    .filter(
    f.col("season").isNotNull() &
    f.col("round").isNotNull() &
    f.col("constructor_id").isNotNull() &
    f.col("driver_id").isNotNull()
)
.dropDuplicates(["season","round","constructor_id","driver_id"])
)

In [0]:
result_final_df=(
    result_valid_df
    .withColumn("race_name",f.initcap(f.col("race_name")))
)

In [0]:
result_final_df.show()

In [0]:
(
    result_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)